# RLHF Clinical Red-Teaming — Pipeline Demo
**Audrey Tjokro · Stephen Dong · Niki Karanikola**

End-to-end demo of all three methods (`baseline` / `dpo` / `ppo`) followed by a test-split evaluation. Every run uses **drastically reduced** hyperparameters — just enough to exercise the full pipeline (data → models → train → eval → artifact sync) so you can sanity-check the wiring before launching paper-grade runs.

This is a **driver only**: no training logic lives here. Each section shells out to `python -m src <method> ...` with an `OVERRIDES` block that shrinks the run to a few minutes.

**Total expected wall-time on a Colab A100:** ~15–25 min for all four sections.

## 1. Mount Drive (optional — used as a local cache for HF + checkpoints)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone repo at a specific commit/branch
Pin a SHA for paper-grade reproducibility. Pass `BRANCH = 'main'` for HEAD.

In [2]:
REPO_URL = 'https://github.com/stephendongg/rlhf-clinical-redteaming.git'
BRANCH   = 'main'
REPO_DIR = '/content/rlhf-clinical-redteaming'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())
%cd $REPO_DIR

0868d78872cb6cd1d661f204d4b5bb4ddeee9fc6
/content/rlhf-clinical-redteaming


## 3. Install requirements

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

## 4. Secrets + GCS auth
Add `OPENAI_API_KEY` to Colab Secrets (left sidebar → key icon). The demo overrides each run to use the OpenAI judge (`gpt-4o-mini`) for speed.

In [4]:
from google.colab import userdata, auth
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()

# GCS auth — uses your Google account; bucket gs://results_043026 must be in project rlhf-clinical-redteaming.
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'rlhf-clinical-redteaming'

## 5. Shared demo settings + run helper

`TINY_DATA` shrinks the seed splits to a handful of scenarios. `TINY_GEN` shortens conversations and generations. Each method below adds its own training-loop shrink overrides on top.

If you want to scale up, edit just these dicts (or the per-method `OVERRIDES`).

In [5]:
GCS_BUCKET = 'gs://results_043026'

# Tiny seed splits — full configs default to 100/50/100.
TINY_DATA = ['data.n_train=4', 'data.n_dev=4','data.n_test=4']

# Shorter conversations + generations — full configs default to 5 turns / 256 tokens.
TINY_GEN = ['max_turns=2', 'attacker_max_new_tokens=128', 'target_max_new_tokens=128']

# Use the OpenAI judge for speed; HF judge adds ~5 GB VRAM and a model load.
DEMO_JUDGE = ['judge_backend=openai', 'judge_model=gpt-4o-mini']

EXTRA_FLAGS = ['--allow-dirty']

In [6]:
import shlex, subprocess, time, os

os.makedirs('logs', exist_ok=True)

def run_method(method: str, run_name: str, overrides: list, use_test: bool = False) -> int:
    """Run `python -m src <method> ...`; full log saved to logs/, compact summary printed."""
    cmd = ['python', '-m', 'src', method,
           '--config', f'configs/{method}.yaml',
           '--run-name', run_name,
           '--gcs-bucket', GCS_BUCKET]
    for o in overrides:
        cmd += ['--override', o]
    if use_test:
        cmd.append('--use-test')
    cmd += EXTRA_FLAGS

    print('$', ' '.join(shlex.quote(c) for c in cmd))
    print('=' * 80)
    start = time.time()
    log = open(f'logs/{run_name}.log', 'w')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        if any(k in line for k in ('->', 'Final', 'Done.', 'Synced', 'Finished', 'error')):
            print(line, end='')
    log.close()
    rc = proc.wait()
    elapsed = (time.time() - start) / 60
    print('=' * 80)
    print(f'Finished {method!r} run_name={run_name!r} rc={rc} in {elapsed:.1f} min')
    if rc != 0:
        raise RuntimeError(f'{method} run failed with return code {rc}')
    return rc

## 6. Baseline (dev split)

Untuned attacker vs. target on the dev split. No training; this is the floor that DPO / PPO are trying to beat. Reports ASR, TTF, and effectiveness for n=4 scenarios.

In [8]:
run_method(method='baseline', run_name='demo-baseline-dev', overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE)

$ python -m src baseline --config configs/baseline.yaml --run-name demo-baseline-dev --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --allow-dirty
baseline eval:  75%|███████▌  | 3/4 [01:16<00:17, 17.85s/seed, asr=0.50, eff=0.35]19:29:17 INFO    redteam_rlhf.baseline |   -> success=False eff=0.000 running_ASR=0.500
19:29:17 INFO    redteam_rlhf.baseline | Final baseline metrics: {'n_trials': 4, 'n_successes': 2, 'ASR': 0.5, 'avg_TTF_successes_only': 2.0, 'avg_effectiveness': 0.35, 'split': 'dev'}
19:29:17 INFO    redteam_rlhf.gcs | GCS upload: results/runs/93fb44eb7560/split_fingerprint.jsonl -> gs://results_043026/baseline/93fb44eb7560/split_fingerprint.jsonl
19:29:20 INFO    redteam_rlhf.gcs | GCS upload: results/runs/93fb44eb7560/trajectories.jsonl -> gs://resu

0

## 7. DPO (1 outer iter, dev eval)

Iterative DPO with the smallest possible loop:
- 1 outer iteration (collect → train → eval) instead of 4
- 2 rollouts per seed instead of 4
- 1 epoch over cached pairs instead of 2
- gradient_accum=1

A LoRA adapter is saved under the run dir and synced to GCS.

In [9]:
DPO_TINY = ['dpo.n_outer=1', 'dpo.n_per_seed=2','dpo.n_epochs=1', 'dpo.grad_accum=1',]

run_method(method='dpo', run_name='demo-dpo-dev', overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + DPO_TINY)

$ python -m src dpo --config configs/dpo.yaml --run-name demo-dpo-dev --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --override dpo.n_outer=1 --override dpo.n_per_seed=2 --override dpo.n_epochs=1 --override dpo.grad_accum=1 --allow-dirty
19:35:59 INFO    redteam_rlhf.dpo | Final DPO metrics: {'n_trials': 4, 'n_successes': 1, 'ASR': 0.25, 'avg_TTF_successes_only': 1.0, 'avg_effectiveness': 0.175, 'split': 'dev'}
19:36:00 INFO    redteam_rlhf.gcs | GCS upload: results/runs/70da4e251571/split_fingerprint.jsonl -> gs://results_043026/dpo/70da4e251571/split_fingerprint.jsonl
19:36:01 INFO    redteam_rlhf.gcs | GCS upload: results/runs/70da4e251571/training_log.jsonl -> gs://results_043026/dpo/70da4e251571/training_log.jsonl
19:36:02 INFO    redteam_rlhf.gcs | GCS uplo

0

## 8. PPO (2 train steps, dev eval)

PPO with the smallest possible loop:
- 2 training steps instead of 100
- 2 conversations per update instead of 4
- gradient_accumulation_steps=1
- checkpoint_every=2 (so we get one checkpoint at the end)

Final-step adapter is saved under the run dir and synced to GCS.

In [10]:
PPO_TINY = ['ppo.n_train_steps=2', 'ppo.n_convos_per_update=2', 'ppo.gradient_accumulation_steps=1', 'ppo.checkpoint_every=2']

run_method(method='ppo', run_name='demo-ppo-dev', overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + PPO_TINY)

$ python -m src ppo --config configs/ppo.yaml --run-name demo-ppo-dev --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --override ppo.n_train_steps=2 --override ppo.n_convos_per_update=2 --override ppo.gradient_accumulation_steps=1 --override ppo.checkpoint_every=2 --allow-dirty
19:37:43 INFO    redteam_rlhf.ppo | PPO training complete. Final running ASR: 0.000
19:37:46 INFO    redteam_rlhf.gcs | GCS upload: results/runs/bd8adfd3161f/split_fingerprint.jsonl -> gs://results_043026/ppo/bd8adfd3161f/split_fingerprint.jsonl
19:37:47 INFO    redteam_rlhf.gcs | GCS upload: results/runs/bd8adfd3161f/severity_metrics.png -> gs://results_043026/ppo/bd8adfd3161f/severity_metrics.png
19:37:48 INFO    redteam_rlhf.gcs | GCS upload: results/runs/bd8adfd3161f/training_log.jsonl 

0

## 9. Test-split evaluation

Adding `--use-test` to any method flips the final eval from the dev split to the held-out test split. For `dpo` / `ppo` this re-runs training as well — that's how the system is wired (training and final eval happen in the same invocation). For paper-grade numbers you'd run each method **once** with `--use-test` directly.

Below we just demo the test-eval flag on `baseline` (no training, so it's fast).

In [11]:
run_method(method='baseline', run_name='demo-baseline-test', overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE, use_test=True)

$ python -m src baseline --config configs/baseline.yaml --run-name demo-baseline-test --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --use-test --allow-dirty
baseline eval:  75%|███████▌  | 3/4 [01:15<00:19, 19.98s/seed, asr=0.50, eff=0.35]19:40:02 INFO    redteam_rlhf.baseline |   -> success=False eff=0.000 running_ASR=0.500
19:40:02 INFO    redteam_rlhf.baseline | Final baseline metrics: {'n_trials': 4, 'n_successes': 2, 'ASR': 0.5, 'avg_TTF_successes_only': 2.0, 'avg_effectiveness': 0.35, 'split': 'test'}
19:40:02 INFO    redteam_rlhf.gcs | GCS upload: results/runs/78ebe28edea8/split_fingerprint.jsonl -> gs://results_043026/baseline/78ebe28edea8/split_fingerprint.jsonl
19:40:03 INFO    redteam_rlhf.gcs | GCS upload: results/runs/78ebe28edea8/trajectories.jsonl

0

If you want to test-evaluate the trained DPO / PPO attackers as part of this demo, uncomment below. Each will retrain with the tiny knobs and then eval on the test split.

In [ ]:
run_method(method='dpo', run_name='demo-dpo-test', overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + DPO_TINY, use_test=True)

run_method(method='ppo', run_name='demo-ppo-test', overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + PPO_TINY, use_test=True)

## 10. Inspect this demo's artifacts in GCS

In [13]:
!gsutil ls -r {GCS_BUCKET}/baseline/ | grep demo- | tail -20
!gsutil ls -r {GCS_BUCKET}/dpo/      | grep demo- | tail -20
!gsutil ls -r {GCS_BUCKET}/ppo/      | grep demo- | tail -20

Caught CTRL-C (signal 2) - exiting
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe
^C
